In [ ]:
# Import required libraries and define global hyperparameters

import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
import random
from tqdm import tqdm
import time
import copy
import matplotlib.pyplot as plt
import numpy as np
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# PARAM
TARGET_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10

# --- KAGGLE CONFIGURATION ---
# Replace with your actual credentials for the professor
#os.environ['KAGGLE_USERNAME'] = "your_username"
#os.environ['KAGGLE_KEY'] = "your_api_key"


In [ ]:
# Utilities for dataset integrity check, train/val/test splitting, and Kaggle dataset setup

def separator(title=""):
    if title:
        print(f"\n{'─' * 10} {title} {'─' * 10}")
    else:
        print(f"{'─' * 30}")

def validate_and_load_dataset(base_path, train_img_path, train_map_path, val_img_path, val_map_path, create_test=True, test_size=0.5):
    try:
        if not base_path.exists():
            raise FileNotFoundError(f"Base path does not exist: {base_path}")

        # LOAD PATHS
        train_images = sorted(list(train_img_path.glob("*.jpg")) + list(train_img_path.glob("*.png")))
        train_maps = sorted(list(train_map_path.glob("*.jpg")) + list(train_map_path.glob("*.png")))

        val_images = sorted(list(val_img_path.glob("*.jpg")) + list(val_img_path.glob("*.png")))
        val_maps = sorted(list(val_map_path.glob("*.jpg")) + list(val_map_path.glob("*.png")))

        # CHECKS
        assert len(train_images) == len(train_maps), "Train mismatch images/maps"
        assert len(val_images) == len(val_maps), "Val mismatch images/maps"

        print("Dataset integrity check passed.")

        if create_test:
            val_data = list(zip(val_images, val_maps))

            val_data, test_data = train_test_split(
                val_data,
                test_size=test_size,
                random_state=42
            )

            val_images, val_maps = zip(*val_data)
            test_images, test_maps = zip(*test_data)

            val_images, val_maps = list(val_images), list(val_maps)
            test_images, test_maps = list(test_images), list(test_maps)

            return train_images, train_maps, val_images, val_maps, test_images, test_maps

        return train_images, train_maps, val_images, val_maps

    except (FileNotFoundError, AssertionError) as e:
        print(f"Dataset Error: {e}")
        sys.exit(1)

    except Exception as e:
        print(f"Unexpected error: {e}")
        sys.exit(1)

def setup_dataset(force_redownload=False):
    dataset_slug = "roshan401/salicon"
    destination_folder = Path("salicon_data")

    expected_paths = [
        destination_folder / "images" / "images" / "train",
        destination_folder / "maps" / "train",
    ]

    !pip install -q kaggle

    if force_redownload and destination_folder.exists():
        print("Removing existing dataset...")
        shutil.rmtree(destination_folder)

    dataset_valid = all(path.exists() and any(path.iterdir()) for path in expected_paths)

    if destination_folder.exists() and not dataset_valid:
        print("Dataset folder exists but is incomplete/corrupted. Removing...")
        shutil.rmtree(destination_folder)

    if not dataset_valid:
        print(f"Downloading {dataset_slug} dataset...")
        destination_folder.mkdir(parents=True, exist_ok=True)

        !kaggle datasets download -d {dataset_slug} --unzip -p {destination_folder}

        dataset_valid = all(path.exists() and any(path.iterdir()) for path in expected_paths)

        if dataset_valid:
            print("Download and extraction completed successfully!")
        else:
            raise RuntimeError(
                "Dataset download completed, but required folders are still missing.\n"
                "Check Kaggle credentials, dataset structure, or extraction process."
            )

    else:
        print("Dataset already exists and is valid. Skipping download.")

In [ ]:
# Dataset setup and path configuration

separator("DATASET DOWNLOAD")

setup_dataset()

# --- PATH DEFINITIONS ---
BASE_PATH = Path("salicon_data")
IMAGE_BASE = BASE_PATH / "images" / "images"
MAP_BASE = BASE_PATH / "maps"

# TRAIN PATHS
IMAGE_TRAIN_PATH = IMAGE_BASE / "train"
MAP_TRAIN_PATH = MAP_BASE / "train"

# VALIDATION PATHS
IMAGE_VAL_PATH = IMAGE_BASE / "val"
MAP_VAL_PATH = MAP_BASE / "val"

# ERROR HANDLING
if not IMAGE_TRAIN_PATH.exists() or not MAP_TRAIN_PATH.exists():
    raise NotADirectoryError(
        f"Training Path Error: Folders not found!\n"
        f"Expected: {IMAGE_TRAIN_PATH} and {MAP_TRAIN_PATH}"
    )

if not IMAGE_VAL_PATH.exists() or not MAP_VAL_PATH.exists():
    raise NotADirectoryError(
        f"Validation Path Error: Folders not found!\n"
        f"Expected: {IMAGE_VAL_PATH} and {MAP_VAL_PATH}"
    )

# MODEL SAVE PATH
#SAVE_PATH = Path("saved_models")
#SAVE_PATH.mkdir(exist_ok=True)

from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = Path("/content/drive/MyDrive/saliency_checkpoints")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

UNET_CHECKPOINT_PATH = SAVE_PATH / "unet_checkpoint.pth"
UNET_BEST_MODEL_PATH = SAVE_PATH / "unet_model.pth"

INCEPTION_SE_UNET_CHECKPOINT_PATH = SAVE_PATH / "inceptionSeUnet_checkpoint.pth"
INCEPTION_SE_UNET_BEST_MODEL_PATH = SAVE_PATH / "inceptionSeUnet_model.pth"

separator("CONFIGURATION READY")
print(f"Images Train Path: {IMAGE_TRAIN_PATH}")
print(f"Maps Train Path:   {MAP_TRAIN_PATH}")
print(f"Images Val Path:   {IMAGE_VAL_PATH}")
print(f"Maps Val Path:     {MAP_VAL_PATH}")
print(f"Model Save Path:   {SAVE_PATH}")

In [ ]:
# Device setup

separator("DEVICE SETUP")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(f"Device: {device.type.upper()}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(f"Device: CPU (no GPU found)")

In [ ]:
# Dataset validation

separator("DATASET CONTROLS")

train_images, train_maps, val_images, val_maps, test_images, test_maps = validate_and_load_dataset(base_path=BASE_PATH,
                                                                                                   train_img_path=IMAGE_TRAIN_PATH,
                                                                                                   train_map_path=MAP_TRAIN_PATH,
                                                                                                   val_img_path=IMAGE_VAL_PATH,
                                                                                                   val_map_path=MAP_VAL_PATH)

In [ ]:
# Dataset class definition


class SaliencyDataset(Dataset):
    def __init__(self, image_files, map_files,
                 target_size=(224, 224), split="",
                 transform=None, map_transform=None):
        self.image_files = image_files
        self.map_files = map_files
        self.target_size = (target_size[1], target_size[0])
        self.split = split.lower()
        self.transform = transform if transform else transforms.ToTensor()
        self.map_transform = map_transform if map_transform else transforms.ToTensor()

    def __len__(self):
        return len(self.image_files)

    def __repr__(self):
        split_name = {
            "train": "Train",
            "val": "Val",
            "test": "Test"
        }.get(self.split, "Unknown")

        return f"SaliencyDataset [{split_name}] — {len(self)} samples"

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        image = Image.open(self.image_files[idx]).convert("RGB")
        target = Image.open(self.map_files[idx]).convert("L")

        image = image.resize(self.target_size)
        target = target.resize(self.target_size)

        if self.split == "train" and random.random() > 0.5:
            image = TF.hflip(image)
            target = TF.hflip(target)

        return self.transform(image), self.map_transform(target)

In [ ]:
# Dataset initialization

separator("DATASET INIT")

to_tensor = transforms.Compose([transforms.ToTensor()])
train_dataset = None
val_dataset = None
test_dataset = None

try:
    train_dataset = SaliencyDataset(
        image_files=train_images, map_files=train_maps,
        target_size=TARGET_SIZE, split="train",
        transform=to_tensor, map_transform=to_tensor
    )
    val_dataset = SaliencyDataset(
        image_files=val_images, map_files=val_maps,
        target_size=TARGET_SIZE, split="val",
        transform=to_tensor, map_transform=to_tensor
    )

    if test_images and test_maps:
      test_dataset = SaliencyDataset(
          image_files=test_images, map_files=test_maps,
          target_size=TARGET_SIZE, split="test",
          transform=to_tensor, map_transform=to_tensor
      )

    print(f"Datasets ready!")
    print(f"   {train_dataset}")
    print(f"   {val_dataset}")
    if test_dataset:
      print(f"   {test_dataset}")

except Exception as e:
    print(f"Dataset error: {e}")
    sys.exit(1)

In [ ]:
# Dataloader class initialization

separator("DATALOADER INIT")

pin = device.type == "cuda"
train_loader = None
val_loader = None

try:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=pin)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=pin)
    if test_dataset:
      test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=pin)

    print(f"DataLoaders ready!")
    print(f"   Train: {len(train_loader)} batches")
    print(f"   Val:   {len(val_loader)} batches")
    if test_loader:
        print(f"   Test:   {len(test_loader)} batches")

except Exception as e:
    print(f"DataLoader error: {e}")
    sys.exit(1)

In [ ]:
# Unet + skip connection class definition

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)

        return features, pooled


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.Upsample(
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

        self.conv = DoubleConv(
            in_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([skip, x], dim=1)

        x = self.conv(x)

        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

        self.dec1 = DecoderBlock(1024, 512, 512)
        self.dec2 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec4 = DecoderBlock(128, 64, 64)

        self.head = nn.Conv2d(64, out_channels, kernel_size=1)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            nn.init.xavier_uniform_(module.weight)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec1(x, s4)
        x = self.dec2(x, s3)
        x = self.dec3(x, s2)
        x = self.dec4(x, s1)

        x = self.head(x)

        return torch.sigmoid(x)

In [ ]:
# Unet + skip connection + inception + se class definition

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.global_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class InceptionBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        hidden = out_channels // 4

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.branch_dilated = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.fuse = nn.Sequential(
            nn.Conv2d(hidden * 4, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.se = SEBlock(out_channels)

    def forward(self, x):
        x1 = self.branch1(x)
        x3 = self.branch3(x)
        xd = self.branch_dilated(x)
        xp = self.branch_pool(x)

        x = torch.cat([x1, x3, xd, xp], dim=1)
        x = self.fuse(x)
        x = self.se(x)
        return x

class DoubleConvSkip(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class EncoderBlockSkip(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = DoubleConvSkip(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip_features = self.double_conv(x)
        pooled_output = self.pool(skip_features)
        return skip_features, pooled_output


class DecoderBlockSkip(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.double_conv = DoubleConvSkip(in_channels, out_channels)

    def forward(self, x, skip_connection):
        x = self.up(x)
        x = torch.cat([x, skip_connection], dim=1)
        return self.double_conv(x)


class InceptionSeUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.enc1 = EncoderBlockSkip(in_channels, 64)
        self.enc2 = EncoderBlockSkip(64, 128)
        self.enc3 = EncoderBlockSkip(128, 256)
        self.enc4 = EncoderBlockSkip(256, 512)

        self.bottleneck = InceptionBottleneck(512, 1024)

        self.dec1 = DecoderBlockSkip(1024 + 512, 512)
        self.dec2 = DecoderBlockSkip(512 + 256, 256)
        self.dec3 = DecoderBlockSkip(256 + 128, 128)
        self.dec4 = DecoderBlockSkip(128 + 64, 64)

        self.head = nn.Sequential(
            nn.Conv2d(64, out_channels, 1),
            nn.Sigmoid()
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            torch.nn.init.xavier_uniform_(module.weight)

            if module.bias is not None:
                module.bias.data.zero_()

    def forward(self, x):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec1(x, s4)
        x = self.dec2(x, s3)
        x = self.dec3(x, s2)
        x = self.dec4(x, s1)

        return self.head(x)

In [ ]:
# Neural networks initialization

separator("NEURAL NETWORKS INITIALIZATION")

unet = UNet(in_channels=3, out_channels=1).to(device)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)

print(f"UNet initialized on {device.type.upper()}")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

inceptionSeUNet = InceptionSeUNet(in_channels=3, out_channels=1).to(device)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)

print(f"InceptionSeUNet initialized on {device.type.upper()}")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")


In [ ]:
# Training, evaluation, checkpointing, and visualization utilities

def compute_metrics(outputs, targets, threshold=0.5):
    preds = (outputs > threshold).float()

    intersection = (preds * targets).sum(dim=(1,2,3))
    union = (preds + targets - preds * targets).sum(dim=(1,2,3))

    iou = intersection / (union + 1e-8)

    return iou.mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc="  Training", leave=False)

    for images, maps in bar:
        images, maps = images.to(device), maps.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, maps)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    iou_total = 0.0

    with torch.no_grad():
        for images, maps in loader:
            images, maps = images.to(device), maps.to(device)

            outputs = model(images)
            loss = criterion(outputs, maps)

            running_loss += loss.item() * images.size(0)

            iou = compute_metrics(outputs, maps)
            iou_total += iou * images.size(0)

    n = len(loader.dataset)

    return running_loss / n, iou_total / n

def plot_comparison(images, targets, outputs):
    img = images[0].cpu().permute(1, 2, 0).numpy()

    gt = targets[0].cpu().squeeze().numpy()

    pred = outputs[0].cpu().squeeze().numpy()

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title("Immagine Input")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(gt, cmap='jet')
    plt.title("Ground Truth (Target)")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred, cmap='jet')
    plt.title("Model prediction")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

def test(model, test_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    iou_total = 0.0

    last_images = None
    last_maps = None
    last_outputs = None

    bar = tqdm(test_loader, desc="Testing", leave=False)

    with torch.no_grad():
        for images, maps in bar:
            images, maps = images.to(device), maps.to(device)

            outputs = model(images)
            loss = criterion(outputs, maps)

            running_loss += loss.item() * images.size(0)

            iou = compute_metrics(outputs, maps)

            iou_total += iou * images.size(0)

            bar.set_postfix(
                loss=f"{loss.item():.4f}",
                iou=f"{iou:.3f}"
            )

            last_images = images
            last_maps = maps
            last_outputs = outputs

    n = len(test_loader.dataset)
    test_loss = running_loss / n
    test_iou = iou_total / n

    if last_images is not None:
        print("\nQualitative comparison:")
        plot_comparison(last_images, last_maps, last_outputs)

    return test_loss, test_iou


def load_checkpoint(model, optimizer, checkpoint_path, device, useCheckpoint=True):

    if checkpoint_path.exists() and useCheckpoint:
        print(f"\nLoading checkpoint from {checkpoint_path}")

        checkpoint = torch.load(checkpoint_path, map_location=device)

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        start_epoch = checkpoint["epoch"] + 1
        best_val_loss = checkpoint["best_val_loss"]

        train_loss_history = checkpoint["train_loss_history"]
        val_loss_history = checkpoint["val_loss_history"]

        patience_counter = checkpoint.get("patience_counter", 0)

        print(f"Resuming from epoch {start_epoch}")
        print(f"Best val loss: {best_val_loss:.4f}")

        return start_epoch, best_val_loss, train_loss_history, val_loss_history, patience_counter

    else:
        print("\nNo checkpoint found, starting from scratch.")
        return 0, float("inf"), [], [], 0

def train(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    save_path,
    checkpoint_path,
    patience=5,
    useCheckpoint=True
):

    start_epoch, best_val_loss, train_loss_history, val_loss_history, patience_counter = load_checkpoint(
        model, optimizer, checkpoint_path, device, useCheckpoint
    )

    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, num_epochs):

        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_iou = validate(model, val_loader, criterion, device)

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"IoU: {val_iou:.4f}")

        # BEST MODEL
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, save_path)
            patience_counter = 0
        else:
            patience_counter += 1

        # CHECKPOINT
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "train_loss_history": train_loss_history,
            "val_loss_history": val_loss_history,
            "patience_counter": patience_counter
        }, checkpoint_path)

        # EARLY STOPPING
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    model.load_state_dict(best_model_wts)

    return train_loss_history, val_loss_history

def plot_losses(train1, val1, train2=None, val2=None, name1="Model 1", name2="Model 2"):
    epochs = range(1, len(train1) + 1)

    plt.figure(figsize=(10, 5))

    plt.plot(epochs, train1, label=f"{name1} Train")
    plt.plot(epochs, val1, label=f"{name1} Val")

    if train2 is not None:
        plt.plot(epochs, train2, label=f"{name2} Train")
        plt.plot(epochs, val2, label=f"{name2} Val")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Comparison")

    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Loss function and optimizer definitions

criterion = nn.MSELoss()
print(f"Defined Loss Function: {type(criterion)}")

# Unet
unetOptimizer = optim.Adam(unet.parameters(), lr=LEARNING_RATE)
print(f"Defined Optimizer fro unet: {type(unetOptimizer)} with initial LR={LEARNING_RATE}")

# InceptionSeUNet
inceptionSeUNetOptimizer = optim.Adam(inceptionSeUNet.parameters(), lr=LEARNING_RATE)
print(f"Defined Optimizer for inceptionSeUNet: {type(inceptionSeUNetOptimizer)} with initial LR={LEARNING_RATE}")

In [ ]:
# Training the networks

separator("TRAINING")

# Unet
unet_train_history, unet_val_history = train(
    model=unet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=unetOptimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    save_path=UNET_BEST_MODEL_PATH,
    checkpoint_path=UNET_CHECKPOINT_PATH,
    useCheckpoint=True
)

# InceptionSeUNet
inceptionSeUNet_train_history, inceptionSeUNet_val_history = train(
    model=inceptionSeUNet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=inceptionSeUNetOptimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    save_path=INCEPTION_SE_UNET_BEST_MODEL_PATH,
    checkpoint_path=INCEPTION_SE_UNET_CHECKPOINT_PATH,
    useCheckpoint=True
)

In [ ]:
# Testing the networks

separator("TESTING")

# Unet
unet_test_loss, unet_test_iou = test(
    model=unet,
    test_loader=test_loader,
    criterion=criterion,
    device=device
)

# InceptionSeUNet
inceptionSeUNet_test_loss, inceptionSeUNet_test_iou = test(
    model=inceptionSeUNet,
    test_loader=test_loader,
    criterion=criterion,
    device=device
)

In [ ]:
# Training-validation loss plot

separator("TRAINING-VALIDATION LOSS")

plot_losses(
    unet_train_history,
    unet_val_history,
    inceptionSeUNet_train_history,
    inceptionSeUNet_val_history,
    name1="UNet",
    name2="Inception-SE UNet"
)

In [ ]:
# Some test stats

separator("TEST STATS")

results = pd.DataFrame({
    "Model": ["UNet", "Inception-SE UNet"],

    "Loss": [
        unet_test_loss,
        inceptionSeUNet_test_loss
    ],

    "IoU": [
        unet_test_iou,
        inceptionSeUNet_test_iou
    ],

    "% IoU Gain": [
        0.0,
        (inceptionSeUNet_test_iou - unet_test_iou) / unet_test_iou * 100
    ],

    "% Loss Reduction": [
        0.0,
        (unet_test_loss - inceptionSeUNet_test_loss) / unet_test_loss * 100
    ]
})

print(results)

In [ ]:
# Test stats visualization

models = ["UNet", "Inception-SE UNet"]

loss = [unet_test_loss, inceptionSeUNet_test_loss]
iou = [unet_test_iou, inceptionSeUNet_test_iou]

x = range(len(models))

plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 3, 1)
plt.bar(x, loss)
plt.xticks(x, models, rotation=15)
plt.title("Test Loss")

# IoU
plt.subplot(1, 3, 2)
plt.bar(x, iou)
plt.xticks(x, models, rotation=15)
plt.title("IoU")

plt.tight_layout()
plt.show()